# 句読点のない文章に読点と句点を付ける（句読点の復元）

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/mamorlis/nlpcookbook/blob/main/notebooks/03_japanese_tokenization/punctuation_restoration.ipynb)

NLP クックブック 03-28 ／ 日本語の形態素解析と文分割

> GPU ランタイム推奨（ランタイム → ランタイムのタイプを変更 → T4 GPU）

音声認識の書き起こしのように句読点のない日本語の文章に、読点「、」と句点「。」を付けたい。句読点を消した文章から学習データを作り、文字単位の BERT で「各文字の後ろに何が来るか」を当てる。

In [1]:
!pip install -q fugashi unidic-lite

In [2]:
import random, re
from datasets import load_dataset

random.seed(0)
PUNCT = {"、": 1, "。": 2}          # ラベル: 0 = なし、1 = 後ろに読点、2 = 後ろに句点
NAMES = ["なし", "読点", "句点"]
WIN = 200                           # 1 件あたりの文字数 (句読点を除く)

def strip_punct(text):
    """句読点を消した文字列と、各文字の後ろに付いていた句読点のラベルを返す"""
    chars, labels = [], []
    for c in text:
        if c in PUNCT:
            if labels:
                labels[-1] = PUNCT[c]
        else:
            chars.append(c); labels.append(0)
    return "".join(chars), labels

# 日本語 Wikipedia (wikimedia/wikipedia の 20231101.ja、CC BY-SA 3.0 / GFDL) をストリーミングで読む
# 記事の本文の段落をつなぎ、句読点を消してから WIN 文字ずつに切る (切れ目は文の境界とは限らない)
wiki = load_dataset("wikimedia/wikipedia", "20231101.ja", split="train", streaming=True)
train, test, test_doc = [], [], []      # test_doc: 評価用の各件がどの記事から来たか
for i, article in enumerate(wiki):
    if any(w in article["title"] for w in ["一覧", "曖昧さ回避"]):
        continue
    paras = [l.strip() for l in article["text"].split("\n")
             if len(l) >= 40 and l.strip().endswith("。") and not re.search(r"[，．]", l)]   # 「，．」を使う記事は除く
    chars, labels = strip_punct("".join(paras))
    dest = test if i % 10 == 0 else train            # 記事単位で学習用と評価用に分ける
    for s in range(0, min(len(chars), 4 * WIN) - WIN + 1, WIN):   # 1 記事から最大 4 件
        dest.append((chars[s:s + WIN], labels[s:s + WIN]))
        if dest is test:
            test_doc.append(i)
    if len(train) >= 20000 and len(test) >= 1000:
        break
train, test, test_doc = train[:20000], test[:1000], test_doc[:1000]
print(len(train), len(test))
print(test[0][0][:80])
print("読点の割合:", round(sum(l.count(1) for _, l in train) / (WIN * len(train)), 3),
      " 句点の割合:", round(sum(l.count(2) for _, l in train) / (WIN * len(train)), 3))

20000 1000
アンパサンド（&, ）は並立助詞「…と…」を意味する記号であるラテン語で「…と…」を表す接続詞 "et" の合字を起源とする現代のフォントでもTrebuchet
読点の割合: 0.028  句点の割合: 0.019


In [3]:
import torch, unicodedata
from transformers import (AutoModelForTokenClassification, AutoTokenizer, DataCollatorForTokenClassification,
                          Trainer, TrainingArguments, set_seed)

set_seed(0)
# 東北大の文字単位の日本語 BERT (Apache 2.0)。語彙は 1 文字ずつなので、文字とトークンが 1 対 1 に対応する
model_name = "tohoku-nlp/bert-base-japanese-char-v3"
tokenizer = AutoTokenizer.from_pretrained(model_name)
vocab = tokenizer.get_vocab()

def encode(chars):
    # tokenizer() を通すと MeCab で単語に切ってから文字に分けるが、ここでは 1 文字ずつ直接 ID にして、文字とラベルの位置をそろえる。
    # 語彙は NFKC 正規化済み (全角の「（」「Ａ」「１」はなく、半角の「(」「A」「1」がある) なので、1 文字ずつ NFKC にしてから引く。
    # NFKC で複数の文字になるもの (「㍿」など) や語彙にない文字は [UNK]
    return ([tokenizer.cls_token_id] + [vocab.get(unicodedata.normalize("NFKC", c), tokenizer.unk_token_id) for c in chars]
            + [tokenizer.sep_token_id])

def to_features(data):
    return [{"input_ids": encode(c), "labels": [-100] + l + [-100]} for c, l in data]   # [CLS] と [SEP] は損失から除く

model = AutoModelForTokenClassification.from_pretrained(
    model_name, num_labels=3, id2label=dict(enumerate(NAMES)), label2id={n: i for i, n in enumerate(NAMES)})
BF16 = torch.cuda.get_device_capability()[0] >= 8      # bfloat16 は Ampere (A100, L4) 以降。T4 では float16 を使う
args = TrainingArguments(
    output_dir="punct-out", num_train_epochs=1, learning_rate=5e-5, warmup_steps=30, weight_decay=0.01,
    per_device_train_batch_size=32, lr_scheduler_type="linear", bf16=BF16, fp16=not BF16,
    logging_steps=100, save_strategy="no", report_to="none", seed=0)
trainer = Trainer(model=model, args=args, train_dataset=to_features(train),
                  data_collator=DataCollatorForTokenClassification(tokenizer))
trainer.train()   # T4 で 10 分程度

Step,Training Loss
100,0.199403
200,0.038625
300,0.038175
400,0.035022
500,0.033606
600,0.033686


TrainOutput(global_step=625, training_loss=0.06189576416015625, metrics={'train_runtime': 88.1691, 'train_samples_per_second': 226.837, 'train_steps_per_second': 7.089, 'total_flos': 2061813361680000.0, 'train_loss': 0.06189576416015625, 'epoch': 1.0})

In [4]:
import numpy as np
from sklearn.metrics import precision_recall_fscore_support

model.eval()

@torch.no_grad()
def predict(texts, batch_size=64):
    """句読点のない文字列 (各 WIN 文字以下) ごとに、各文字の後ろのラベルの列を返す"""
    out = []
    for b in range(0, len(texts), batch_size):
        batch = DataCollatorForTokenClassification(tokenizer)([{"input_ids": encode(t)} for t in texts[b:b + batch_size]])
        logits = model(**{k: v.to(model.device) for k, v in batch.items()}).logits.float()
        out += [row[1:len(t) + 1].argmax(-1).tolist() for t, row in zip(texts[b:b + batch_size], logits)]
    return out

def report(gold, pred, title):
    g, p = sum(gold, []), sum(pred, [])
    prec, rec, f1, n = precision_recall_fscore_support(g, p, labels=[1, 2], zero_division=0)
    print(title)
    for k, name in enumerate(["読点", "句点"]):
        print(f"  {name}: 適合率 {prec[k]:.3f}  再現率 {rec[k]:.3f}  F1 {f1[k]:.3f}  (正解 {n[k]} 個)")

def restore(chars, labels):
    return "".join(c + ("", "、", "。")[l] for c, l in zip(chars, labels))

pred = predict([c for c, _ in test])
report([l for _, l in test], pred, f"BERT (評価 {len(test)} 件、{WIN * len(test)} 文字)")
ids = sum((encode(c)[1:-1] for c, _ in test), [])
print(f"評価データで [UNK] になった文字の割合: {ids.count(tokenizer.unk_token_id) / len(ids):.4f}")
print("正解:", restore(*test[1])[:120])
print("予測:", restore(test[1][0], pred[1])[:120])

BERT (評価 1000 件、200000 文字)
  読点: 適合率 0.830  再現率 0.751  F1 0.789  (正解 5592 個)
  句点: 適合率 0.945  再現率 0.963  F1 0.954  (正解 3849 個)
評価データで [UNK] になった文字の割合: 0.0005
正解: per se（それ自体）を用いて "A per se A" のように唱えられていた。また、アルファベットの最後に、27番目の文字のように "&" を加えることも広く行われていた。"&" はラテン語で et と読まれていたが、のちに英語で a
予測: per se（それ自体）を用いて "A per se A" のように唱えられていた。また、アルファベットの最後に27番目の文字のように "&" を加えることも広く行われていた。"&" はラテン語で et と読まれていたが、のちに英語で an


In [5]:
# 書き起こし風の話し言葉に付けてみる。長い文章は WIN 文字ずつに切って予測する
def punctuate(text):
    text = re.sub(r"[\s、。]", "", text)
    pieces = [text[s:s + WIN] for s in range(0, len(text), WIN)]
    return "".join(restore(p, l) for p, l in zip(pieces, predict(pieces)))

speech = ("えーと今日はですね新しい製品の発表についてお話ししたいと思いますまず価格ですが"
          "従来モデルと同じに据え置きました次に発売日ですが来月の十日を予定しております"
          "ご質問があればあとでまとめてお受けします")
restored = punctuate(speech)
print(restored)
# 句点が付けば文に分けられる。ここでは簡単のため句点だけで分ける (括弧の扱いはこの章の「日本語の文を句点で分割する (括弧内は分割しない)」を参照)
for s in re.findall(r"[^。]+。?", restored):
    print("-", s)

えーと今日はですね、新しい製品の発表についてお話ししたいと思います。まず価格ですが従来モデルと同じに据え置きました。次に発売日ですが、来月の十日を予定しております。ご質問があればあとでまとめてお受けします。
- えーと今日はですね、新しい製品の発表についてお話ししたいと思います。
- まず価格ですが従来モデルと同じに据え置きました。
- 次に発売日ですが、来月の十日を予定しております。
- ご質問があればあとでまとめてお受けします。


In [6]:
# LLM に句読点を付けさせて比べる (sarashina2.2-3b-instruct、MIT ライセンス)。評価用の先頭 100 件だけを使う
import difflib
import sentencepiece as spm
from huggingface_hub import hf_hub_download
from tokenizers import models
from transformers import AutoModelForCausalLM

name = "sbintuitions/sarashina2.2-3b-instruct-v0.1"
tok = AutoTokenizer.from_pretrained(name, padding_side="left")
# transformers 5 系は、このモデルの SentencePiece (Unigram) を BPE に変換してしまい、学習時と違う分割になる。
# 元の tokenizer.model から語彙とスコアを読み、分割の方式だけを Unigram に戻す (特殊トークンやチャットテンプレートはそのまま)
sp = spm.SentencePieceProcessor(model_file=hf_hub_download(name, "tokenizer.model"))
tok.backend_tokenizer.model = models.Unigram(
    [(sp.id_to_piece(i), sp.get_score(i)) for i in range(sp.get_piece_size())], unk_id=sp.unk_id(), byte_fallback=True)
llm = AutoModelForCausalLM.from_pretrained(name, dtype=torch.bfloat16 if BF16 else torch.float16, device_map="auto").eval()

INST = "次の文章に読点「、」と句点「。」を付けてください。文字の追加・削除・変更はせず、句読点を付けた文章だけを出力してください。\n\n"
example = train[0]   # 1-shot の例 (学習データから)
def messages(text):
    return [{"role": "user", "content": INST + example[0]}, {"role": "assistant", "content": restore(*example)},
            {"role": "user", "content": INST + text}]

@torch.no_grad()
def llm_punctuate(texts, batch_size=10):
    outs = []
    for b in range(0, len(texts), batch_size):
        enc = tok.apply_chat_template([messages(t) for t in texts[b:b + batch_size]], add_generation_prompt=True,
                                      padding=True, return_tensors="pt", return_dict=True).to(llm.device)
        gen = llm.generate(**enc, max_new_tokens=300, do_sample=False, pad_token_id=tok.pad_token_id)
        outs += tok.batch_decode(gen[:, enc["input_ids"].shape[1]:], skip_special_tokens=True)
    return [o.strip() for o in outs]

def project(src, out):
    """LLM の出力の句読点を、元の文字列 src の文字位置に写す (LLM が文字を変えた部分は句読点なし扱い)"""
    o_chars, o_labels = strip_punct(out)
    sm = difflib.SequenceMatcher(None, src, o_chars, autojunk=False)
    labels = [0] * len(src)
    for a, b, n in sm.get_matching_blocks():
        labels[a:a + n] = o_labels[b:b + n]
    changes = [(src[i1:i2], o_chars[j1:j2]) for op, i1, i2, j1, j2 in sm.get_opcodes() if op != "equal"]
    return labels, changes

sub = test[:100]
outs = llm_punctuate([c for c, _ in sub])   # T4 では 5 分以上かかる見込み (モデルのダウンロードは別)
proj = [project(c, o) for (c, _), o in zip(sub, outs)]
print("句読点以外の文字を変えなかった出力:", sum(not ch for _, ch in proj), "/", len(sub))
print("LLM が変えた箇所の例 (元 → 出力):", [ch for _, chs in proj[:5] for ch in chs[:2]])
report([l for _, l in sub], [l for l, _ in proj], "LLM (1-shot、100 件)")
report([l for _, l in sub], pred[:100], "BERT (同じ 100 件)")
print("LLM:", outs[1][:120])
print(f"GPU メモリの最大使用量: {torch.cuda.max_memory_allocated() / 1e9:.1f} GB")

句読点以外の文字を変えなかった出力: 1 / 100
LLM が変えた箇所の例 (元 → 出力): [(' ', '単語が起源となっていることが多い'), ('', 'よ'), ('', 'になった'), ('', 'た'), ('', "`&'（"), ('', '）'), ('', ' '), ('', 'という特徴を持つ')]
LLM (1-shot、100 件)
  読点: 適合率 0.751  再現率 0.594  F1 0.663  (正解 603 個)
  句点: 適合率 0.890  再現率 0.739  F1 0.808  (正解 395 個)
BERT (同じ 100 件)
  読点: 適合率 0.825  再現率 0.743  F1 0.782  (正解 603 個)
  句点: 適合率 0.939  再現率 0.972  F1 0.955  (正解 395 個)
LLM: per se（それ自体）を用いて "A per se A" のように唱えられていた、またアルファベットの最後に27番目の文字のように "&" を加えることも広く行われていた"&" はラテン語で et と読まれていたがのちに英語で and と
GPU メモリの最大使用量: 9.0 GB


In [7]:
# 100 件は少ないので、ブートストラップで F1 の差の 95% 信頼区間を求める。
# 同じ記事の断片どうしは似ているので、断片ではなく記事を単位に復元抽出する
from sklearn.metrics import f1_score

gold100, bert100, llm100 = [l for _, l in sub], pred[:100], [l for l, _ in proj]
docs = sorted(set(test_doc[:len(sub)]))
by_doc = {d: [j for j in range(len(sub)) if test_doc[j] == d] for d in docs}
print("100 件のもとの記事の数:", len(docs))
rng = np.random.default_rng(0)
diffs = {1: [], 2: []}
for _ in range(1000):
    idx = [j for d in rng.choice(docs, len(docs)) for j in by_doc[d]]
    g = sum((gold100[i] for i in idx), [])
    fb = f1_score(g, sum((bert100[i] for i in idx), []), labels=[1, 2], average=None, zero_division=0)
    fl = f1_score(g, sum((llm100[i] for i in idx), []), labels=[1, 2], average=None, zero_division=0)
    for k in (1, 2):
        diffs[k].append(fb[k - 1] - fl[k - 1])
for k in (1, 2):
    lo, hi = np.percentile(diffs[k], [2.5, 97.5])
    print(f"{NAMES[k]}の F1 の差 (BERT − LLM): 95% 信頼区間 [{lo:.3f}, {hi:.3f}]")

100 件のもとの記事の数: 27
読点の F1 の差 (BERT − LLM): 95% 信頼区間 [0.086, 0.157]
句点の F1 の差 (BERT − LLM): 95% 信頼区間 [0.107, 0.187]


- 句読点を消すだけで、正解付きの学習データがいくらでも作れる（自己教師あり）。付録「仕組みを理解するための実装」の「単語分割を文字単位の系列ラベリングとして解く」と同じく、文字ごとのラベル付けとして解いている。語彙が 1 文字単位の BERT を使うと、トークンと文字の位置合わせが要らない。ただしこの BERT の語彙は NFKC 正規化済みで、全角の「（」「１」などを含まないので、1 文字ずつ NFKC にしてから ID を引いている（評価データで [UNK] になった文字は 0.05%）。
- 評価用の 1000 件（20 万文字）で、句点は F1 0.954（適合率 0.945、再現率 0.963）と高く、読点は F1 0.789（適合率 0.830、再現率 0.751）と低かった。読点の位置は書き手によって揺れ、付けても付けなくてもよい箇所が多い（上の例でも「最後に、」の読点を落とした）。元の文章の読点を唯一の正解として数えるので、読点の数値は実用上の質より低めに出ることがある。
- 学習も評価も書き言葉の Wikipedia なので、フィラー（「えーと」）や言い直しを含む実際の書き起こしでの精度は確かめていない。上の話し言葉の例では文末 4 か所すべてに句点が付いたが、「まず価格ですが」の後ろには読点が付かなかった。また、この BERT は事前学習に 2023 年 1 月の日本語 Wikipedia を使っているので、評価用の記事を句読点付きで見ている可能性がある。実際に使うときは、句読点の付いた話し言葉のテキスト（字幕や議事録など）を学習データに加え、自分のデータで評価する。
- 句点を復元すれば、この章の「日本語の文を句点で分割する（括弧内は分割しない）」や「日本語の文分割をライブラリで行って比べる」の方法で文に分けられる。句点の誤りはそのまま文分割の誤りになる。長い文章は 200 文字ずつ重なりなしに切っているので、切れ目の前後の文字は文脈が足りない。重なりを持たせて切り、中央部分の予測だけを使うとよい。
- LLM（sarashina2.2-3b-instruct、1-shot）は、同じ 100 件で読点 F1 0.663、句点 F1 0.808 と BERT（0.782、0.955）より低く、F1 の差のブートストラップ 95% 信頼区間（100 件のもとの 27 記事を単位に復元抽出。読点 [0.086, 0.157]、句点 [0.107, 0.187]）も 0 を含まなかった。句読点以外の文字を変えなかった出力は 100 件中 1 件だけで、文の途中で切れた断片の続きを書き足したり、記号を足したりした。そのため出力を difflib で元の文字列に対応づけてから評価している。より大きな LLM や API のモデルでの結果は確かめていない。
- GPU メモリの最大使用量は、BERT の学習と 3B の LLM の推論を合わせて 9.0 GB で、T4 (16GB) に収まる。
- データとモデルの出典とライセンス: 日本語 Wikipedia（wikimedia/wikipedia、CC BY-SA 3.0 / GFDL）、tohoku-nlp/bert-base-japanese-char-v3（東北大学、Apache 2.0）、sbintuitions/sarashina2.2-3b-instruct-v0.1（SB Intuitions、MIT）。sarashina のトークナイザーを直すコードについては、大規模言語モデルの章の「オープンな LLM とチャットする（チャットテンプレート）」の注意書きを参照。